In [2]:
import logging
import os
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logger = logging.getLogger("01_eda")

ModuleNotFoundError: No module named 'pandas'

## 1. CALABARZON Province PSGC Reference

In [ ]:
# Official PSGC codes for CALABARZON provinces
CALABARZON_PROVINCES = {
    "PH040100000": "Batangas",
    "PH040200000": "Cavite",
    "PH040300000": "Laguna",
    "PH040400000": "Quezon",
    "PH040500000": "Rizal",
}

# Expected quarters: 2020Q1 – 2025Q4 (24 quarters)
expected_quarters = [
    f"{year}_Q{q}"
    for year in range(2020, 2026)
    for q in range(1, 5)
]

print(f"CALABARZON provinces: {len(CALABARZON_PROVINCES)}")
print(f"Expected quarters: {len(expected_quarters)} ({expected_quarters[0]} – {expected_quarters[-1]})")
pd.DataFrame(
    [{"psgc_code": k, "province": v} for k, v in CALABARZON_PROVINCES.items()]
)

## 2. Run PSA Fetcher (Auto-download from psa.gov.ph + rsso04a.psa.gov.ph)

In [ ]:
from app.ml.corpus.psa_fetcher import fetch_psa_reports

psa_records = fetch_psa_reports(
    start_date="2020-01-01",
    end_date="2025-12-31",
)

psa_df = pd.DataFrame(psa_records)
print(f"Total PSA records fetched: {len(psa_df)}")
psa_df.head(10)

## 3. PSA File Coverage — Reports per Report Type

In [ ]:
if not psa_df.empty:
    coverage = psa_df.groupby("report_type").size().rename("count").reset_index()
    print(coverage.to_string(index=False))
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(coverage["report_type"], coverage["count"], color="steelblue")
    ax.set_xlabel("File count")
    ax.set_title("PSA Reports Auto-Fetched by Report Type")
    plt.tight_layout()
    plt.savefig("../data/eval/fig_psa_coverage_by_type.png", dpi=150)
    plt.show()
else:
    print("[WARN] psa_df is empty — PSA fetcher returned no records.")
    print("This is expected if running without live network access.")
    print("In production, run with active internet connection targeting psa.gov.ph.")

## 4. Quarter Coverage Audit — All 24 Quarters Present?

In [ ]:
if not psa_df.empty:
    fetched_quarters = set(psa_df["year_quarter"].unique())
    expected_set = set(expected_quarters)

    present = expected_set & fetched_quarters
    missing = expected_set - fetched_quarters

    print(f"Quarters with data: {len(present)} / {len(expected_quarters)}")
    if missing:
        print(f"\nMISSING quarters (null signal — record in notebook for Chapter III):")
        for q in sorted(missing):
            print(f"  {q}")
    else:
        print("All 24 expected quarters are covered.")
else:
    print("[INFO] Quarter audit skipped — empty PSA DataFrame.")
    print("Expected quarters for CALABARZON 2020Q1–2025Q4:")
    for q in expected_quarters:
        print(f"  {q}")

## 5. Null Count Per Quarter

Logs data gaps that must be acknowledged in Chapter III as limitations.

In [ ]:
if not psa_df.empty:
    null_audit = psa_df.isnull().sum().rename("null_count").reset_index()
    null_audit.columns = ["field", "null_count"]
    print("Null counts per field:")
    print(null_audit.to_string(index=False))

    # Quarter-level null analysis
    quarter_nulls = (
        psa_df.groupby("year_quarter")
        .apply(lambda g: g.isnull().sum().sum())
        .rename("total_nulls")
        .reset_index()
    )
    print("\nTotal null fields per quarter:")
    print(quarter_nulls.to_string(index=False))
else:
    print("[INFO] Null audit skipped — run with live network access.")

## 6. Corpus Raw — Article Counts by Source Domain

In [ ]:
corpus_path = Path("../data/raw/corpus_raw.parquet")

if corpus_path.exists():
    corpus_df = pd.read_parquet(corpus_path)
    print(f"corpus_raw.parquet: {len(corpus_df):,} articles")
    print(f"Columns: {list(corpus_df.columns)}")
    display(corpus_df.head())

    # Article counts per source domain
    domain_counts = (
        corpus_df["source_domain"]
        .value_counts()
        .rename("article_count")
        .reset_index()
    )
    domain_counts.columns = ["source_domain", "article_count"]
    print("\nArticle counts by source domain:")
    print(domain_counts.to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(domain_counts["source_domain"], domain_counts["article_count"], color="teal")
    ax.set_xlabel("Article count")
    ax.set_title("Corpus Raw — Articles per Credible Source Domain")
    plt.tight_layout()
    plt.savefig("../data/eval/fig_corpus_by_domain.png", dpi=150)
    plt.show()
else:
    print(f"[INFO] corpus_raw.parquet not found at {corpus_path}")
    print("Run run_corpus_ingestion() from app.ml.corpus first.")

## 7. PSA Fetcher Output Schema — Handoff to Member B

Member B requires these field names by **end of Week 10** for DB schema and Pydantic contract design.

In [ ]:
schema_reference = pd.DataFrame([
    # --- psa_fetcher.py output ---
    {"source": "psa_fetcher",       "field": "report_type",    "type": "str",   "values": "price_survey | labor_force_survey | poverty_statistics"},
    {"source": "psa_fetcher",       "field": "year_quarter",   "type": "str",   "values": "{YYYY}_Q{N} e.g. 2024_Q2"},
    {"source": "psa_fetcher",       "field": "source_url",     "type": "str",   "values": "PSA domain URL"},
    {"source": "psa_fetcher",       "field": "local_path",     "type": "str",   "values": "data/raw/psa_reports/{type}/{yq}.pdf|html"},
    {"source": "psa_fetcher",       "field": "fetched_at",     "type": "str",   "values": "ISO UTC datetime"},
    {"source": "psa_fetcher",       "field": "http_status",    "type": "int",   "values": "HTTP status code"},
    # --- psa_report_parser.py downstream output (Week 11) ---
    {"source": "psa_report_parser", "field": "province_code",  "type": "str",   "values": "PSGC code e.g. PH040100000"},
    {"source": "psa_report_parser", "field": "quarter",        "type": "str",   "values": "{YYYY}-Q{N} e.g. 2024-Q2"},
    {"source": "psa_report_parser", "field": "food_cpi",       "type": "float", "values": "Food CPI index value (price_survey)"},
    {"source": "psa_report_parser", "field": "rice_price",     "type": "float", "values": "Rice retail price PHP/kg (price_survey)"},
    {"source": "psa_report_parser", "field": "unemployment",   "type": "float", "values": "Unemployment rate % (labor_force_survey)"},
    {"source": "psa_report_parser", "field": "poverty_incidence", "type": "float", "values": "Poverty incidence % (poverty_statistics)"},
    {"source": "psa_report_parser", "field": "population",     "type": "int",   "values": "Municipal/provincial population (poverty_statistics)"},
    # --- corpus_raw.parquet (rss + gnews + wayback) ---
    {"source": "corpus_raw",        "field": "title",          "type": "str",   "values": "Article headline"},
    {"source": "corpus_raw",        "field": "link",           "type": "str",   "values": "Canonical article URL"},
    {"source": "corpus_raw",        "field": "published",      "type": "str",   "values": "ISO UTC datetime string"},
    {"source": "corpus_raw",        "field": "summary",        "type": "str",   "values": "Lead paragraph / RSS description"},
    {"source": "corpus_raw",        "field": "source_domain",  "type": "str",   "values": "Domain from CREDIBLE_DOMAINS"},
])

print("=== PSA Fetcher + Corpus Output Schema (Member A → Member B handoff) ===")
display(schema_reference)

## 8. Chapter III Findings Summary

*(To be updated after running with live network access)*

**Data sources confirmed:**
- PSA auto-fetcher targets: `psa.gov.ph` and `rsso04a.psa.gov.ph`
- Report series: Monthly Price Survey, Regional Labor Force Survey, Full Year Poverty Statistics
- robots.txt compliance: enforced via `urllib.robotparser`
- Domain validation: all URLs validated before download; redirects to non-PSA domains are rejected
- Deduplication: SHA-256 URL-hash prevents re-downloading

**Corpus news sources:**
- RSS: inquirer.net, mb.com.ph, philstar.com, rappler.com, cnnphilippines.com, news.abs-cbn.com, sunstar.com.ph, ptv.gov.ph
- GNews: same CREDIBLE_DOMAINS allowlist enforced
- Wayback: 2020–2022 historical gap-fill; CDX restricted to CREDIBLE_DOMAINS only

**CALABARZON coverage:** All 5 provinces (Batangas, Cavite, Laguna, Quezon, Rizal) × 24 quarters (2020Q1–2025Q4) = 120 province-quarter cells targeted.

**Null / gap findings:** *(Document actual counts here after live run)*
